# Bloque 2: Procesamiento del Lenguaje Natural (NLP)
## Clasificación de Currículos — 5 Modelos + Búsqueda de Hiperparámetros + Producción

Dataset: [Jarvis Calling Hiring Contest — Kaggle](https://www.kaggle.com/competitions/jarvis-calling-hiring-contest/overview)

**Modelos implementados:**
1. RoBERTa (Transformer)
2. Word2Vec + BiLSTM
3. CNN-1D
4. TF-IDF + XGBoost
5. FastText

**Al final:** comparación de métricas y guardado del mejor modelo para producción.

## 0. Instalación de dependencias

In [12]:
# !pip install transformers datasets wordcloud plotly xgboost scikit-learn gensim nltk tensorflow

In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'  

import tensorflow as tf
tf.get_logger().setLevel('ERROR')

## 1. Imports y configuración global

In [14]:
import os, json, random, warnings, pickle
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
import joblib
import kagglehub

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.util import ngrams
from wordcloud import WordCloud

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, ParameterSampler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    roc_curve, precision_recall_curve
)

from gensim.models import Word2Vec
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')

# Semilla global para reproducibilidad
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

TEMPLATE = 'plotly_white'
os.makedirs('saved_models', exist_ok=True)

print('TensorFlow:', tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f'GPU disponible: {[g.name for g in gpus]}')
else:
    print('Usando CPU')

TensorFlow: 2.10.0
Usando CPU


In [15]:
nltk.download('stopwords', quiet=True)
nltk.download('punkt',     quiet=True)
nltk.download('punkt_tab', quiet=True)

True

## 2. Carga y preprocesamiento de datos

In [ ]:
# ─── Ajusta esta ruta a tu archivo local ───────────────────────────────────
DATA_PATH = r'C:/Users/ASUS/Desktop/Proyecto_Deep_Lenguaje_Natural/data/Resume.csv'
# ───────────────────────────────────────────────────────────────────────────

df = pd.read_csv(DATA_PATH)
df = df[['Resume_str', 'Category']].dropna()
df.index = range(len(df))
print(f'Shape: {df.shape}')
print(f'Categorías ({df.Category.nunique()}):', sorted(df.Category.unique()))
df.head()

Shape: (2484, 2)
Categorías (24): ['ACCOUNTANT', 'ADVOCATE', 'AGRICULTURE', 'APPAREL', 'ARTS', 'AUTOMOBILE', 'AVIATION', 'BANKING', 'BPO', 'BUSINESS-DEVELOPMENT', 'CHEF', 'CONSTRUCTION', 'CONSULTANT', 'DESIGNER', 'DIGITAL-MEDIA', 'ENGINEERING', 'FINANCE', 'FITNESS', 'HEALTHCARE', 'HR', 'INFORMATION-TECHNOLOGY', 'PUBLIC-RELATIONS', 'SALES', 'TEACHER']


,Resume_str,Category
0,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,HR
1,"HR SPECIALIST, US HR OPERATIONS ...",HR
2,HR DIRECTOR Summary Over 2...,HR
3,HR SPECIALIST Summary Dedica...,HR
4,HR MANAGER Skill Highlights ...,HR


In [17]:
# ─── Preprocesamiento de texto ──────────────────────────────────────────────
EXTRA_SW = {'company', 'name', 'city', 'state', 'experience', 'skills'}
STOP_EN  = set(stopwords.words('english')) | EXTRA_SW

def clean_text(text: str) -> str:
    """Limpia texto crudo de PDF: quita HTML, números, puntuación y stopwords."""
    text = re.sub(r'[\n\r\t]+', ' ', text)
    text = re.sub(r' {2,}',     ' ', text)
    text = re.sub(r'/',         ' ', text)
    text = text.lower().strip()
    text = re.sub(r'[^a-z\s]', '', text)
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t not in STOP_EN and len(t) > 2]
    return ' '.join(tokens)

df['clean_text']   = df['Resume_str'].apply(clean_text)
df['clean_tokens'] = df['clean_text'].apply(str.split)
df['word_count']   = df['clean_tokens'].apply(len)
print('Preprocesamiento listo.')
df[['Category','clean_text','word_count']].head(3)

Preprocesamiento listo.


,Category,clean_text,word_count
0,HR,administrator marketing associate administrato...,450
1,HR,specialist operations summary versatile media ...,484
2,HR,director summary years recruiting plus years h...,663


## 3. Codificación de etiquetas y división train / val / test

In [18]:
le = LabelEncoder()
df['label'] = le.fit_transform(df['Category'])
N_CLASSES   = len(le.classes_)
print(f'Número de clases: {N_CLASSES}')

# 70 % train | 15 % val | 15 % test (estratificado)
X_train_raw, X_temp, y_train, y_temp = train_test_split(
    df['Resume_str'].values, df['label'].values,
    test_size=0.30, random_state=SEED, stratify=df['label']
)
X_val_raw, X_test_raw, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50, random_state=SEED, stratify=y_temp
)

# Versión limpia para modelos que la necesitan
X_train_clean = [clean_text(t).split() for t in X_train_raw]
X_val_clean   = [clean_text(t).split() for t in X_val_raw]
X_test_clean  = [clean_text(t).split() for t in X_test_raw]

# Versión string limpia (para TF-IDF y FastText char-ngrams)
X_train_str = [' '.join(t) for t in X_train_clean]
X_val_str   = [' '.join(t) for t in X_val_clean]
X_test_str  = [' '.join(t) for t in X_test_clean]

print(f'Train: {len(X_train_raw)} | Val: {len(X_val_raw)} | Test: {len(X_test_raw)}')

Número de clases: 24
Train: 1738 | Val: 373 | Test: 373


## 4. Utilidades comunes

In [19]:
# ─── Registro global de métricas ────────────────────────────────────────────
RESULTS = {}

def compute_metrics(y_true, y_pred, y_proba, model_name: str) -> dict:
    """Calcula y almacena métricas de clasificación."""
    metrics = dict(
        accuracy  = accuracy_score(y_true, y_pred),
        precision = precision_score(y_true, y_pred, average='weighted', zero_division=0),
        recall    = recall_score(y_true, y_pred,    average='weighted', zero_division=0),
        f1        = f1_score(y_true, y_pred,        average='weighted', zero_division=0),
        roc_auc   = roc_auc_score(y_true, y_proba,  multi_class='ovr',  average='weighted'),
    )
    RESULTS[model_name] = metrics
    print(f'\n===== {model_name} =====')
    for k, v in metrics.items():
        print(f'  {k:12s}: {v:.4f}')
    return metrics

def plot_training_curves(history_dict: dict, title: str):
    """Curvas loss / accuracy del historial de Keras."""
    h   = history_dict
    eps = list(range(1, len(h['loss']) + 1))
    fig = make_subplots(rows=1, cols=2, subplot_titles=('Loss', 'Accuracy'))
    fig.add_trace(go.Scatter(x=eps, y=h['loss'],         name='Train Loss',
                             line=dict(color='royalblue')),          row=1, col=1)
    fig.add_trace(go.Scatter(x=eps, y=h['val_loss'],     name='Val Loss',
                             line=dict(color='tomato')),             row=1, col=1)
    fig.add_trace(go.Scatter(x=eps, y=h['accuracy'],     name='Train Acc',
                             line=dict(color='royalblue', dash='dot')), row=1, col=2)
    fig.add_trace(go.Scatter(x=eps, y=h['val_accuracy'], name='Val Acc',
                             line=dict(color='tomato',    dash='dot')), row=1, col=2)
    fig.update_layout(title=f'{title} — Curvas de entrenamiento',
                      template=TEMPLATE, height=400)
    fig.show()

def plot_confusion_matrix(y_true, y_pred, title: str):
    """Heatmap de matriz de confusión."""
    cm  = confusion_matrix(y_true, y_pred)
    fig = px.imshow(cm,
                    labels=dict(x='Predicho', y='Real', color='Conteo'),
                    x=le.classes_, y=le.classes_,
                    color_continuous_scale='Blues',
                    title=f'{title} — Matriz de confusión',
                    text_auto=True)
    fig.update_layout(template=TEMPLATE, height=700, xaxis_tickangle=-45)
    fig.show()

def plot_roc_pr(y_true, y_proba, title: str):
    """Curvas ROC y Precision-Recall por clase."""
    fig_roc = go.Figure()
    fig_pr  = go.Figure()
    for i, cls in enumerate(le.classes_):
        bin_y   = (y_true == i).astype(int)
        fpr, tpr, _ = roc_curve(bin_y, y_proba[:, i])
        auc_i       = roc_auc_score(bin_y, y_proba[:, i])
        fig_roc.add_trace(go.Scatter(x=fpr, y=tpr,
                                     name=f'{cls} (AUC={auc_i:.2f})', mode='lines'))
        prec_c, rec_c, _ = precision_recall_curve(bin_y, y_proba[:, i])
        fig_pr.add_trace(go.Scatter(x=rec_c, y=prec_c, name=cls, mode='lines'))

    fig_roc.add_shape(type='line', x0=0, y0=0, x1=1, y1=1,
                      line=dict(dash='dash', color='gray'))
    fig_roc.update_layout(title=f'{title} — Curva ROC',
                          xaxis_title='FPR', yaxis_title='TPR',
                          template=TEMPLATE, height=500)
    fig_roc.show()
    fig_pr.update_layout(title=f'{title} — Curva Precision-Recall',
                         xaxis_title='Recall', yaxis_title='Precision',
                         template=TEMPLATE, height=500)
    fig_pr.show()

def tokens_to_seq(tokens, word2idx, max_len):
    idx = [word2idx.get(t, 1) for t in tokens][:max_len]
    return idx + [0] * (max_len - len(idx))

def build_sequences(token_lists, word2idx, max_len):
    return np.array([tokens_to_seq(t, word2idx, max_len) for t in token_lists])

print('Utilidades listas.')

Utilidades listas.


---
## Modelo 1 — RoBERTa
Transformer preentrenado con fine-tuning. Se prueban distintas learning rates y longitudes de secuencia.

In [20]:
from transformers import RobertaTokenizer, TFRobertaForSequenceClassification

# ─── Búsqueda de hiperparámetros (grid pequeño) ─────────────────────────────
ROBERTA_GRID = [
    {'max_len': 128, 'lr': 2e-5, 'batch': 8, 'epochs': 6},
    {'max_len': 128, 'lr': 3e-5, 'batch': 8, 'epochs': 6},
    {'max_len': 256, 'lr': 2e-5, 'batch': 4, 'epochs': 4},
    {'max_len': 128, 'lr': 2e-5, 'batch': 8, 'epochs': 10}
]

roberta_tokenizer = RobertaTokenizer.from_pretrained('roberta-base')

def make_roberta_dataset(texts, labels, max_len, batch_size, shuffle=False):
    enc = roberta_tokenizer(
        list(texts), max_length=max_len, padding='max_length',
        truncation=True, return_tensors='tf'
    )
    ds = tf.data.Dataset.from_tensor_slices((
        {'input_ids': enc['input_ids'], 'attention_mask': enc['attention_mask']},
        labels
    ))
    if shuffle:
        ds = ds.shuffle(500, seed=SEED)
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

roberta_search_results = []

for cfg in ROBERTA_GRID:
    print(f'\n[RoBERTa] config: {cfg}')
    train_ds = make_roberta_dataset(X_train_raw, y_train, cfg['max_len'], cfg['batch'], shuffle=True)
    val_ds   = make_roberta_dataset(X_val_raw,   y_val,   cfg['max_len'], cfg['batch'])

    mdl = TFRobertaForSequenceClassification.from_pretrained(
        'roberta-base', num_labels=N_CLASSES
    )
    mdl.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=cfg['lr']),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=['accuracy']
    )
    cb_list = [
        callbacks.EarlyStopping(monitor='val_loss', patience=2,
                                restore_best_weights=True, verbose=0),
    ]
    h = mdl.fit(train_ds, validation_data=val_ds,
                epochs=cfg['epochs'], callbacks=cb_list, verbose=1)
    best_vl = min(h.history['val_loss'])
    print(f'  => Best Val Loss: {best_vl:.4f}')
    roberta_search_results.append({'cfg': cfg, 'val_loss': best_vl, 'model': mdl, 'history': h})

best_roberta_run = min(roberta_search_results, key=lambda x: x['val_loss'])
best_roberta_cfg = best_roberta_run['cfg']
print(f'\nMejor config RoBERTa: {best_roberta_cfg}')


[RoBERTa] config: {'max_len': 128, 'lr': 2e-05, 'batch': 8, 'epochs': 6}


Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFRobertaForSequenceClassification: ['roberta.embeddings.position_ids']
- This IS expected if you are initializing TFRobertaForSequenceClassification from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFRobertaForSequenceClassification from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
Some weights or buffers of the TF 2.0 model TFRobertaForSequenceClassification were not initialized from the PyTorch model and are newly initialized: ['classifier.dense.weight', 'classifier.dense.bias', 'classifier.out_proj.weight', 'classifier.out_proj.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predicti

Epoch 1/6
218/218 [==============================] - 441s 2s/step - loss: 2.5928 - accuracy: 0.3550 - val_loss: 1.4394 - val_accuracy: 0.6729
Epoch 2/6
218/218 [==============================] - 407s 2s/step - loss: 1.2132 - accuracy: 0.7261 - val_loss: 1.0848 - val_accuracy: 0.7346
Epoch 3/6
218/218 [==============================] - 407s 2s/step - loss: 0.8862 - accuracy: 0.7848 - val_loss: 0.9460 - val_accuracy: 0.7399
Epoch 4/6
218/218 [==============================] - 416s 2s/step - loss: 0.7133 - accuracy: 0.8234 - val_loss: 0.9542 - val_accuracy: 0.7507
Epoch 5/6
218/218 [==============================] - 401s 2s/step - loss: 0.5880 - accuracy: 0.8579 - val_loss: 1.0209 - val_accuracy: 0.7480
  => Best Val Loss: 0.9460

[RoBERTa] config: {'max_len': 128, 'lr': 3e-05, 'batch': 8, 'epochs': 6}


Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFRobertaForSequenceClassification: ['roberta.embeddings.position_ids']
- This IS expected if you are initializing TFRobertaForSequenceClassification from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFRobertaForSequenceClassification from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
Some weights or buffers of the TF 2.0 model TFRobertaForSequenceClassification were not initialized from the PyTorch model and are newly initialized: ['classifier.dense.weight', 'classifier.dense.bias', 'classifier.out_proj.weight', 'classifier.out_proj.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predicti

Epoch 1/6
218/218 [==============================] - 1136s 5s/step - loss: 2.2055 - accuracy: 0.4465 - val_loss: 1.3267 - val_accuracy: 0.6702
Epoch 2/6
218/218 [==============================] - 959s 4s/step - loss: 1.0768 - accuracy: 0.7348 - val_loss: 1.0090 - val_accuracy: 0.7265
Epoch 3/6
218/218 [==============================] - 946s 4s/step - loss: 0.8348 - accuracy: 0.7888 - val_loss: 0.9663 - val_accuracy: 0.7319
Epoch 4/6
218/218 [==============================] - 732s 3s/step - loss: 0.7040 - accuracy: 0.8228 - val_loss: 1.0349 - val_accuracy: 0.7319
Epoch 5/6
218/218 [==============================] - 744s 3s/step - loss: 0.6139 - accuracy: 0.8389 - val_loss: 1.0137 - val_accuracy: 0.7560
  => Best Val Loss: 0.9663

[RoBERTa] config: {'max_len': 256, 'lr': 2e-05, 'batch': 4, 'epochs': 4}


Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFRobertaForSequenceClassification: ['roberta.embeddings.position_ids']
- This IS expected if you are initializing TFRobertaForSequenceClassification from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFRobertaForSequenceClassification from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
Some weights or buffers of the TF 2.0 model TFRobertaForSequenceClassification were not initialized from the PyTorch model and are newly initialized: ['classifier.dense.weight', 'classifier.dense.bias', 'classifier.out_proj.weight', 'classifier.out_proj.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predicti

Epoch 1/4
435/435 [==============================] - 2315s 5s/step - loss: 2.1308 - accuracy: 0.4753 - val_loss: 1.3024 - val_accuracy: 0.6568
Epoch 2/4
435/435 [==============================] - 2859s 7s/step - loss: 1.0735 - accuracy: 0.7514 - val_loss: 0.9251 - val_accuracy: 0.7721
Epoch 3/4
435/435 [==============================] - 2560s 6s/step - loss: 0.8030 - accuracy: 0.8067 - val_loss: 0.9108 - val_accuracy: 0.7587
Epoch 4/4
435/435 [==============================] - 2848s 7s/step - loss: 0.6800 - accuracy: 0.8285 - val_loss: 0.9519 - val_accuracy: 0.7507
  => Best Val Loss: 0.9108

[RoBERTa] config: {'max_len': 128, 'lr': 2e-05, 'batch': 8, 'epochs': 10}


Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFRobertaForSequenceClassification: ['roberta.embeddings.position_ids']
- This IS expected if you are initializing TFRobertaForSequenceClassification from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFRobertaForSequenceClassification from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
Some weights or buffers of the TF 2.0 model TFRobertaForSequenceClassification were not initialized from the PyTorch model and are newly initialized: ['classifier.dense.weight', 'classifier.dense.bias', 'classifier.out_proj.weight', 'classifier.out_proj.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predicti

Epoch 1/10
218/218 [==============================] - 752s 3s/step - loss: 2.5141 - accuracy: 0.3659 - val_loss: 1.4203 - val_accuracy: 0.6649
Epoch 2/10
218/218 [==============================] - 730s 3s/step - loss: 1.1870 - accuracy: 0.7307 - val_loss: 1.0390 - val_accuracy: 0.7587
Epoch 3/10
218/218 [==============================] - 665s 3s/step - loss: 0.9198 - accuracy: 0.7773 - val_loss: 0.9901 - val_accuracy: 0.7373
Epoch 4/10
218/218 [==============================] - 686s 3s/step - loss: 0.7386 - accuracy: 0.8199 - val_loss: 0.9640 - val_accuracy: 0.7399
Epoch 5/10
218/218 [==============================] - 843s 4s/step - loss: 0.6272 - accuracy: 0.8377 - val_loss: 0.9681 - val_accuracy: 0.7587
Epoch 6/10
218/218 [==============================] - 1151s 5s/step - loss: 0.5055 - accuracy: 0.8711 - val_loss: 1.0274 - val_accuracy: 0.7560
  => Best Val Loss: 0.9640

Mejor config RoBERTa: {'max_len': 256, 'lr': 2e-05, 'batch': 4, 'epochs': 4}


In [21]:
# ─── Evaluación del mejor RoBERTa ───────────────────────────────────────────
best_roberta = best_roberta_run['model']
test_ds_r    = make_roberta_dataset(X_test_raw, y_test,
                                    best_roberta_cfg['max_len'],
                                    best_roberta_cfg['batch'])

logits_list, rob_labels = [], []
for bx, by in test_ds_r:
    logits_list.append(best_roberta(bx, training=False).logits.numpy())
    rob_labels.extend(by.numpy())

rob_logits = np.concatenate(logits_list)
rob_proba  = tf.nn.softmax(rob_logits).numpy()
rob_preds  = np.argmax(rob_proba, axis=1)
rob_labels = np.array(rob_labels)

compute_metrics(rob_labels, rob_preds, rob_proba, 'RoBERTa')
plot_training_curves(best_roberta_run['history'].history, 'RoBERTa')
plot_confusion_matrix(rob_labels, rob_preds, 'RoBERTa')
plot_roc_pr(rob_labels, rob_proba, 'RoBERTa')

# Guardar pesos
best_roberta.save_weights('saved_models/roberta_best.weights.h5')
print('Pesos RoBERTa guardados.')


===== RoBERTa =====
  accuracy    : 0.7909
  precision   : 0.8040
  recall      : 0.7909
  f1          : 0.7801
  roc_auc     : 0.9665


Pesos RoBERTa guardados.


---
## Modelo 2 — Word2Vec + BiLSTM
Embeddings entrenados en el corpus + red BiLSTM. Random Search con 12 combinaciones.

In [23]:
def train_w2v(token_lists, dim):
    return Word2Vec(sentences=token_lists, vector_size=dim,
                    window=5, min_count=2, workers=4, epochs=10, seed=SEED)

def build_bilstm(vocab_size, emb_dim, emb_matrix,
                 lstm_units, num_layers, dropout, n_cls, lr):
    inp = tf.keras.Input(shape=(None,), dtype='int32')
    x   = layers.Embedding(vocab_size, emb_dim,
                            weights=[emb_matrix],
                            trainable=True, mask_zero=True)(inp)
    x   = layers.Dropout(dropout)(x)
    for i in range(num_layers):
        ret_seq = (i < num_layers - 1)
        x = layers.Bidirectional(
            layers.LSTM(lstm_units, return_sequences=ret_seq,
                        dropout=dropout, recurrent_dropout=0.1)
        )(x)
        x = layers.Dropout(dropout)(x)
    out = layers.Dense(n_cls, activation='softmax')(x)
    mdl = models.Model(inp, out)
    mdl.compile(optimizer=tf.keras.optimizers.Adam(lr),
                loss='sparse_categorical_crossentropy',
                metrics=['accuracy'])
    return mdl

# ─── Random Search ────────────────────────────────────────────────────────
BILSTM_GRID = {
    'lstm_units':   [32, 64, 128],
    'num_layers':   [1, 2],
    'dropout':      [0.3, 0.5],   # era dropout_rate
    'lr':           [0.0005, 0.001, 0.003],   # era learning_rate
    'batch_size':   [32, 64],
    'seq_len':      [50, 100],    # era sequence_len
    'emb_dim':      [100, 200],   # era embedding_dim
    'epochs':       [20, 50],
}

N_ITER_BILSTM = 12

bilstm_results = []
all_tokens_train = X_train_clean + X_val_clean

for i, p in enumerate(ParameterSampler(BILSTM_GRID, n_iter=N_ITER_BILSTM,
                                        random_state=SEED)):
    print(f'\n[BiLSTM {i+1}/{N_ITER_BILSTM}] {p}')
    w2v      = train_w2v(all_tokens_train, p['emb_dim'])
    word2idx = {w: i+2 for i, w in enumerate(w2v.wv.key_to_index)}
    vocab_sz = len(word2idx) + 2
    emb_mat  = np.zeros((vocab_sz, p['emb_dim']))
    for w, idx in word2idx.items():
        emb_mat[idx] = w2v.wv[w]

    Xtr = build_sequences(X_train_clean, word2idx, p['seq_len'])
    Xvl = build_sequences(X_val_clean,   word2idx, p['seq_len'])

    mdl = build_bilstm(vocab_sz, p['emb_dim'], emb_mat,
                       p['lstm_units'], p['num_layers'],
                       p['dropout'], N_CLASSES, p['lr'])
    cb_es = callbacks.EarlyStopping(monitor='val_loss', patience=5,
                                    restore_best_weights=True, verbose=0)
    h = mdl.fit(Xtr, y_train, validation_data=(Xvl, y_val),
                epochs=p['epochs'], batch_size=p['batch_size'],
                callbacks=[cb_es], verbose=0)
    best_vl = min(h.history['val_loss'])
    print(f'  => Best Val Loss: {best_vl:.4f}')
    bilstm_results.append({
        'params': p, 'val_loss': best_vl,
        'word2idx': word2idx, 'emb_mat': emb_mat, 'vocab_sz': vocab_sz
    })

best_bilstm_run = min(bilstm_results, key=lambda x: x['val_loss'])
best_bp = best_bilstm_run['params']
print(f'\nMejores hiperparámetros BiLSTM: {best_bp}')
print(f'Mejor Val Loss: {best_bilstm_run["val_loss"]:.4f}')


[BiLSTM 1/12] {'seq_len': 50, 'num_layers': 2, 'lstm_units': 64, 'lr': 0.001, 'epochs': 20, 'emb_dim': 200, 'dropout': 0.5, 'batch_size': 32}
  => Best Val Loss: 1.2214

[BiLSTM 2/12] {'seq_len': 50, 'num_layers': 2, 'lstm_units': 128, 'lr': 0.0005, 'epochs': 50, 'emb_dim': 200, 'dropout': 0.3, 'batch_size': 32}
  => Best Val Loss: 1.1381

[BiLSTM 3/12] {'seq_len': 50, 'num_layers': 2, 'lstm_units': 128, 'lr': 0.001, 'epochs': 50, 'emb_dim': 100, 'dropout': 0.3, 'batch_size': 64}
  => Best Val Loss: 1.1103

[BiLSTM 4/12] {'seq_len': 50, 'num_layers': 2, 'lstm_units': 64, 'lr': 0.003, 'epochs': 50, 'emb_dim': 100, 'dropout': 0.5, 'batch_size': 64}
  => Best Val Loss: 1.3150

[BiLSTM 5/12] {'seq_len': 50, 'num_layers': 2, 'lstm_units': 64, 'lr': 0.0005, 'epochs': 50, 'emb_dim': 200, 'dropout': 0.3, 'batch_size': 64}
  => Best Val Loss: 1.1290

[BiLSTM 6/12] {'seq_len': 50, 'num_layers': 1, 'lstm_units': 64, 'lr': 0.0005, 'epochs': 50, 'emb_dim': 200, 'dropout': 0.3, 'batch_size': 64}
  

In [24]:
# ─── Entrenamiento final BiLSTM ──────────────────────────────────────────────
br = best_bilstm_run
Xtr_b = build_sequences(X_train_clean, br['word2idx'], best_bp['seq_len'])
Xvl_b = build_sequences(X_val_clean,   br['word2idx'], best_bp['seq_len'])
Xte_b = build_sequences(X_test_clean,  br['word2idx'], best_bp['seq_len'])

bilstm_model = build_bilstm(
    br['vocab_sz'], best_bp['emb_dim'], br['emb_mat'],
    best_bp['lstm_units'], best_bp['num_layers'],
    best_bp['dropout'], N_CLASSES, best_bp['lr']
)
cb_bilstm = [
    callbacks.EarlyStopping(monitor='val_loss', patience=5,
                            restore_best_weights=True, verbose=1),
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3,
                                min_lr=1e-6, verbose=1),
]
hist_bilstm = bilstm_model.fit(
    Xtr_b, y_train, validation_data=(Xvl_b, y_val),
    epochs=best_bp['epochs'], batch_size=best_bp['batch_size'],
    callbacks=cb_bilstm
)

bl_proba = bilstm_model.predict(Xte_b, batch_size=best_bp['batch_size'])
bl_preds = np.argmax(bl_proba, axis=1)

compute_metrics(y_test, bl_preds, bl_proba, 'Word2Vec+BiLSTM')
plot_training_curves(hist_bilstm.history, 'Word2Vec + BiLSTM')
plot_confusion_matrix(y_test, bl_preds, 'Word2Vec + BiLSTM')
plot_roc_pr(y_test, bl_proba, 'Word2Vec + BiLSTM')

# Guardar modelo Keras + artefactos de preprocesamiento
bilstm_model.save('saved_models/bilstm_best.keras')
with open('saved_models/bilstm_word2idx.pkl', 'wb') as f:
    pickle.dump(br['word2idx'], f)
print('BiLSTM guardado.')

Epoch 1/20
55/55 [==============================] - 21s 314ms/step - loss: 2.5285 - accuracy: 0.3400 - val_loss: 1.6354 - val_accuracy: 0.5791 - lr: 0.0030
Epoch 2/20
55/55 [==============================] - 16s 298ms/step - loss: 1.5123 - accuracy: 0.6139 - val_loss: 1.2300 - val_accuracy: 0.6729 - lr: 0.0030
Epoch 3/20
55/55 [==============================] - 16s 298ms/step - loss: 1.1652 - accuracy: 0.7043 - val_loss: 1.1204 - val_accuracy: 0.6810 - lr: 0.0030
Epoch 4/20
55/55 [==============================] - 16s 298ms/step - loss: 0.9717 - accuracy: 0.7457 - val_loss: 1.0774 - val_accuracy: 0.7051 - lr: 0.0030
Epoch 5/20
55/55 [==============================] - 16s 298ms/step - loss: 0.8450 - accuracy: 0.7756 - val_loss: 1.0290 - val_accuracy: 0.7373 - lr: 0.0030
Epoch 6/20
55/55 [==============================] - 16s 299ms/step - loss: 0.7133 - accuracy: 0.8044 - val_loss: 1.0714 - val_accuracy: 0.7158 - lr: 0.0030
Epoch 7/20
55/55 [==============================] - 16s 298ms/st

BiLSTM guardado.


---
## Modelo 3 — CNN-1D
Convoluciones paralelas con distintos tamaños de kernel (Text-CNN). Random Search con 10 combinaciones.

In [26]:
# ✅ Vocabulario compartido para CNN (estable y ordenado)
all_tr_tokens = [t for tokens in X_train_clean for t in tokens]
vocab_counter  = Counter(all_tr_tokens)

cnn_vocab = {'<PAD>': 0, '<UNK>': 1}
for i, (w, c) in enumerate(sorted(vocab_counter.items())):
    if c >= 2:
        cnn_vocab[w] = len(cnn_vocab)  # índice incremental seguro

CNN_VOCAB_SIZE = len(cnn_vocab)
print(f'Vocabulario CNN: {CNN_VOCAB_SIZE:,} tokens')

def build_sequences_cnn(token_lists, word2idx, max_len):
    unk_idx = word2idx.get('<UNK>', 1)
    seqs = []
    for tokens in token_lists:
        ids = [word2idx.get(t, unk_idx) for t in tokens]
        ids = ids[:max_len]
        ids += [0] * (max_len - len(ids))
        seqs.append(ids)
    return np.array(seqs, dtype='int32')

def build_cnn1d(vocab_size, emb_dim, n_filters,
                kernel_sizes, dropout, n_cls, lr, max_len):
    inp    = tf.keras.Input(shape=(max_len,), dtype='int32')
    x      = layers.Embedding(vocab_size, emb_dim, mask_zero=False)(inp)
    x      = layers.Dropout(dropout)(x)
    pooled = []
    for k in kernel_sizes:
        c = layers.Conv1D(n_filters, k, activation='relu', padding='valid')(x)
        pooled.append(layers.GlobalMaxPooling1D()(c))
    concat = layers.Concatenate()(pooled)
    concat = layers.Dropout(dropout)(concat)
    out    = layers.Dense(n_cls, activation='softmax')(concat)
    mdl    = models.Model(inp, out)
    mdl.compile(optimizer=tf.keras.optimizers.Adam(lr),
                loss='sparse_categorical_crossentropy',
                metrics=['accuracy'])
    return mdl

# ─── Random Search CNN ────────────────────────────────────────────────────
CNN_GRID = {
    'emb_dim':      [64, 100, 150],
    'n_filters':    [64, 128, 256],
    'kernel_sizes': [[2,3,4], [3,4,5], [2,3,4,5]],
    'dropout':      [0.3, 0.4, 0.5],
    'lr':           [5e-4, 1e-3, 3e-3],
    'batch_size':   [32, 64],
    'max_len':      [75, 100, 150],
    'epochs':       [30],
}
N_ITER_CNN = 10
cnn_results = []

for i, p in enumerate(ParameterSampler(CNN_GRID, n_iter=N_ITER_CNN, random_state=SEED)):
    print(f'\n[CNN {i+1}/{N_ITER_CNN}] {p}')
    Xtr_c = build_sequences_cnn(X_train_clean, cnn_vocab, p['max_len'])
    Xvl_c = build_sequences_cnn(X_val_clean,   cnn_vocab, p['max_len'])

    # Verificación de seguridad
    assert Xtr_c.max() < CNN_VOCAB_SIZE, f"Índice fuera de rango en train: {Xtr_c.max()}"
    assert Xvl_c.max() < CNN_VOCAB_SIZE, f"Índice fuera de rango en val: {Xvl_c.max()}"

    mdl = build_cnn1d(CNN_VOCAB_SIZE, p['emb_dim'], p['n_filters'],
                      p['kernel_sizes'], p['dropout'], N_CLASSES,
                      p['lr'], p['max_len'])
    cb_es = callbacks.EarlyStopping(monitor='val_loss', patience=5,
                                    restore_best_weights=True, verbose=0)
    h = mdl.fit(Xtr_c, y_train, validation_data=(Xvl_c, y_val),
                epochs=p['epochs'], batch_size=p['batch_size'],
                callbacks=[cb_es], verbose=0)
    best_vl = min(h.history['val_loss'])
    print(f'  => Best Val Loss: {best_vl:.4f}')
    cnn_results.append({'params': p, 'val_loss': best_vl})

best_cnn_run = min(cnn_results, key=lambda x: x['val_loss'])
best_cp = best_cnn_run['params']
print(f'\nMejores hiperparámetros CNN-1D: {best_cp}')
print(f'Mejor Val Loss: {best_cnn_run["val_loss"]:.4f}')

Vocabulario CNN: 20,400 tokens

[CNN 1/10] {'n_filters': 128, 'max_len': 75, 'lr': 0.003, 'kernel_sizes': [2, 3, 4, 5], 'epochs': 30, 'emb_dim': 100, 'dropout': 0.4, 'batch_size': 64}
  => Best Val Loss: 1.0875

[CNN 2/10] {'n_filters': 256, 'max_len': 100, 'lr': 0.003, 'kernel_sizes': [3, 4, 5], 'epochs': 30, 'emb_dim': 100, 'dropout': 0.3, 'batch_size': 64}
  => Best Val Loss: 1.0229

[CNN 3/10] {'n_filters': 128, 'max_len': 150, 'lr': 0.003, 'kernel_sizes': [2, 3, 4, 5], 'epochs': 30, 'emb_dim': 64, 'dropout': 0.5, 'batch_size': 64}
  => Best Val Loss: 0.9965

[CNN 4/10] {'n_filters': 256, 'max_len': 100, 'lr': 0.003, 'kernel_sizes': [2, 3, 4, 5], 'epochs': 30, 'emb_dim': 100, 'dropout': 0.4, 'batch_size': 64}
  => Best Val Loss: 1.0445

[CNN 5/10] {'n_filters': 64, 'max_len': 150, 'lr': 0.001, 'kernel_sizes': [3, 4, 5], 'epochs': 30, 'emb_dim': 100, 'dropout': 0.4, 'batch_size': 64}
  => Best Val Loss: 1.0655

[CNN 6/10] {'n_filters': 64, 'max_len': 75, 'lr': 0.003, 'kernel_sizes':

In [27]:
# ─── Entrenamiento final CNN-1D ──────────────────────────────────────────────
Xtr_c = build_sequences(X_train_clean, cnn_vocab, best_cp['max_len'])
Xvl_c = build_sequences(X_val_clean,   cnn_vocab, best_cp['max_len'])
Xte_c = build_sequences(X_test_clean,  cnn_vocab, best_cp['max_len'])

cnn_model = build_cnn1d(CNN_VOCAB_SIZE, best_cp['emb_dim'], best_cp['n_filters'],
                        best_cp['kernel_sizes'], best_cp['dropout'], N_CLASSES,
                        best_cp['lr'], best_cp['max_len'])
cb_cnn = [
    callbacks.EarlyStopping(monitor='val_loss', patience=5,
                            restore_best_weights=True, verbose=1),
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3,
                                min_lr=1e-6, verbose=1),
]
hist_cnn = cnn_model.fit(
    Xtr_c, y_train, validation_data=(Xvl_c, y_val),
    epochs=best_cp['epochs'], batch_size=best_cp['batch_size'],
    callbacks=cb_cnn
)

cnn_proba = cnn_model.predict(Xte_c, batch_size=best_cp['batch_size'])
cnn_preds = np.argmax(cnn_proba, axis=1)

compute_metrics(y_test, cnn_preds, cnn_proba, 'CNN-1D')
plot_training_curves(hist_cnn.history, 'CNN-1D')
plot_confusion_matrix(y_test, cnn_preds, 'CNN-1D')
plot_roc_pr(y_test, cnn_proba, 'CNN-1D')

cnn_model.save('saved_models/cnn1d_best.keras')
with open('saved_models/cnn1d_vocab.pkl', 'wb') as f:
    pickle.dump(cnn_vocab, f)
json.dump(best_cp, open('saved_models/cnn1d_config.json', 'w'), indent=2)
print('CNN-1D guardado.')

Epoch 1/30
55/55 [==============================] - 10s 184ms/step - loss: 3.1631 - accuracy: 0.0483 - val_loss: 3.1286 - val_accuracy: 0.1260 - lr: 5.0000e-04
Epoch 2/30
55/55 [==============================] - 10s 179ms/step - loss: 3.1203 - accuracy: 0.0771 - val_loss: 3.0970 - val_accuracy: 0.2413 - lr: 5.0000e-04
Epoch 3/30
55/55 [==============================] - 10s 178ms/step - loss: 3.0667 - accuracy: 0.1427 - val_loss: 3.0187 - val_accuracy: 0.3539 - lr: 5.0000e-04
Epoch 4/30
55/55 [==============================] - 10s 181ms/step - loss: 2.9149 - accuracy: 0.2860 - val_loss: 2.7763 - val_accuracy: 0.3968 - lr: 5.0000e-04
Epoch 5/30
55/55 [==============================] - 10s 181ms/step - loss: 2.5490 - accuracy: 0.4327 - val_loss: 2.3055 - val_accuracy: 0.4879 - lr: 5.0000e-04
Epoch 6/30
55/55 [==============================] - 10s 181ms/step - loss: 2.0497 - accuracy: 0.5351 - val_loss: 1.8764 - val_accuracy: 0.5603 - lr: 5.0000e-04
Epoch 7/30
55/55 [======================

CNN-1D guardado.


---
## Modelo 4 — TF-IDF + XGBoost
Enfoque clásico de ML. Grid search sobre parámetros de TF-IDF y XGBoost.

In [28]:
# ─── Grid de TF-IDF ────────────────────────────────────────────────────────
TFIDF_GRID = [
    {'max_features': 20000, 'ngram_range': (1,1), 'sublinear_tf': True},
    {'max_features': 30000, 'ngram_range': (1,2), 'sublinear_tf': True},
    {'max_features': 50000, 'ngram_range': (1,2), 'sublinear_tf': True},
]
XGB_GRID = [
    {'n_estimators': 300, 'max_depth': 5, 'lr': 0.1,  'subsample': 0.8},
    {'n_estimators': 500, 'max_depth': 6, 'lr': 0.05, 'subsample': 0.8},
    {'n_estimators': 700, 'max_depth': 7, 'lr': 0.03, 'subsample': 0.9},
]

xgb_results = []

for tfidf_p in TFIDF_GRID:
    tfidf_tmp = TfidfVectorizer(min_df=2, **tfidf_p)
    Xtr_tf = tfidf_tmp.fit_transform(X_train_str)
    Xvl_tf = tfidf_tmp.transform(X_val_str)
    Xte_tf = tfidf_tmp.transform(X_test_str)

    for xgb_p in XGB_GRID:
        print(f'\n[XGB] tfidf={tfidf_p} | xgb={xgb_p}')
        try:
            mdl = XGBClassifier(
                n_estimators=xgb_p['n_estimators'],
                max_depth=xgb_p['max_depth'],
                learning_rate=xgb_p['lr'],
                subsample=xgb_p['subsample'],
                colsample_bytree=0.8,
                eval_metric='mlogloss',
                use_label_encoder=False,
                random_state=SEED,
                n_jobs=-1,
                early_stopping_rounds=30,
                device='cuda'
            )
        except Exception:
            mdl = XGBClassifier(
                n_estimators=xgb_p['n_estimators'],
                max_depth=xgb_p['max_depth'],
                learning_rate=xgb_p['lr'],
                subsample=xgb_p['subsample'],
                colsample_bytree=0.8,
                eval_metric='mlogloss',
                use_label_encoder=False,
                random_state=SEED,
                n_jobs=-1,
                early_stopping_rounds=30
            )
        mdl.fit(Xtr_tf, y_train, eval_set=[(Xvl_tf, y_val)], verbose=0)

        # ── CAMBIO: extraer mlogloss mínimo en lugar de calcular F1 ──
        val_loss = min(mdl.evals_result()['validation_0']['mlogloss'])
        print(f'  => Val Loss (mlogloss): {val_loss:.4f}')

        xgb_results.append({
            'tfidf_p': tfidf_p, 'xgb_p': xgb_p,
            'val_loss': val_loss,   # ← antes era 'val_f1'
            'tfidf': tfidf_tmp,
            'model': mdl, 'Xte': Xte_tf
        })

# ── CAMBIO: min en lugar de max, y key apunta a 'val_loss' ──
best_xgb_run = min(xgb_results, key=lambda x: x['val_loss'])
print(f'\nMejor config TF-IDF: {best_xgb_run["tfidf_p"]}')
print(f'Mejor config XGBoost: {best_xgb_run["xgb_p"]}')
print(f'Mejor Val Loss (mlogloss): {best_xgb_run["val_loss"]:.4f}')


[XGB] tfidf={'max_features': 20000, 'ngram_range': (1, 1), 'sublinear_tf': True} | xgb={'n_estimators': 300, 'max_depth': 5, 'lr': 0.1, 'subsample': 0.8}
  => Val Loss (mlogloss): 4.4828

[XGB] tfidf={'max_features': 20000, 'ngram_range': (1, 1), 'sublinear_tf': True} | xgb={'n_estimators': 500, 'max_depth': 6, 'lr': 0.05, 'subsample': 0.8}
  => Val Loss (mlogloss): 3.5938

[XGB] tfidf={'max_features': 20000, 'ngram_range': (1, 1), 'sublinear_tf': True} | xgb={'n_estimators': 700, 'max_depth': 7, 'lr': 0.03, 'subsample': 0.9}
  => Val Loss (mlogloss): 3.3998

[XGB] tfidf={'max_features': 30000, 'ngram_range': (1, 2), 'sublinear_tf': True} | xgb={'n_estimators': 300, 'max_depth': 5, 'lr': 0.1, 'subsample': 0.8}
  => Val Loss (mlogloss): 5.0629

[XGB] tfidf={'max_features': 30000, 'ngram_range': (1, 2), 'sublinear_tf': True} | xgb={'n_estimators': 500, 'max_depth': 6, 'lr': 0.05, 'subsample': 0.8}
  => Val Loss (mlogloss): 3.6100

[XGB] tfidf={'max_features': 30000, 'ngram_range': (1, 2

In [29]:
# ─── Evaluación del mejor XGBoost ───────────────────────────────────────────
best_xgb   = best_xgb_run['model']
best_tfidf = best_xgb_run['tfidf']
Xte_tf     = best_xgb_run['Xte']

xgb_proba = best_xgb.predict_proba(Xte_tf)
xgb_preds = np.argmax(xgb_proba, axis=1)

compute_metrics(y_test, xgb_preds, xgb_proba, 'TF-IDF+XGBoost')

# Curva de pérdida por iteración
evals = best_xgb.evals_result()
iters = list(range(1, len(evals['validation_0']['mlogloss']) + 1))
fig_xgb = go.Figure()
fig_xgb.add_trace(go.Scatter(x=iters, y=evals['validation_0']['mlogloss'],
                              name='Val mlogloss', line=dict(color='tomato')))
fig_xgb.add_vline(x=best_xgb.best_iteration, line_dash='dash', line_color='gray',
                  annotation_text=f'Mejor iter: {best_xgb.best_iteration}')
fig_xgb.update_layout(title='TF-IDF + XGBoost — Loss por iteración',
                      xaxis_title='Iteración', yaxis_title='mlogloss',
                      template=TEMPLATE, height=400)
fig_xgb.show()

plot_confusion_matrix(y_test, xgb_preds, 'TF-IDF + XGBoost')
plot_roc_pr(y_test, xgb_proba, 'TF-IDF + XGBoost')

joblib.dump(best_tfidf, 'saved_models/tfidf_best.joblib')
joblib.dump(best_xgb,   'saved_models/xgboost_best.joblib')
print('TF-IDF + XGBoost guardados con joblib.')


===== TF-IDF+XGBoost =====
  accuracy    : 0.5282
  precision   : 0.4626
  recall      : 0.5282
  f1          : 0.4778
  roc_auc     : 0.8135


TF-IDF + XGBoost guardados con joblib.


---
## Modelo 5 — FastText (n-gramas de caracteres)
Embeddings promediados sobre n-gramas de caracteres. Random Search con 10 combinaciones.

In [30]:
def create_char_ngrams(text, min_n, max_n):
    grams = []
    for n in range(min_n, max_n + 1):
        grams += [text[i:i+n] for i in range(len(text) - n + 1)]
    return grams

def build_ngram_vocab(texts, min_n, max_n, max_feat):
    cnt = Counter()
    for t in texts:
        cnt.update(create_char_ngrams(t, min_n, max_n))
    vocab = {ng: i+2 for i, (ng, _) in enumerate(cnt.most_common(max_feat-2))}
    vocab['<PAD>'] = 0; vocab['<UNK>'] = 1
    return vocab

def text_to_ngram_seq(text, vocab, min_n, max_n, max_len):
    grams = create_char_ngrams(text, min_n, max_n)
    idx   = [vocab.get(g, 1) for g in grams][:max_len]
    return idx + [0] * (max_len - len(idx))

def prepare_ft_seqs(texts, vocab, min_n, max_n, max_len):
    return np.array([text_to_ngram_seq(t, vocab, min_n, max_n, max_len)
                     for t in texts])

def build_fasttext_model(vocab_size, emb_dim, n_cls, lr, max_len):
    inp = tf.keras.Input(shape=(max_len,), dtype='int32')
    x   = layers.Embedding(vocab_size, emb_dim,
                            embeddings_initializer='uniform',
                            mask_zero=False)(inp)
    x   = layers.GlobalAveragePooling1D()(x)
    x   = layers.Dense(128, activation='relu')(x)
    x   = layers.Dropout(0.3)(x)
    out = layers.Dense(n_cls, activation='softmax')(x)
    mdl = models.Model(inp, out)
    mdl.compile(optimizer=tf.keras.optimizers.Adam(lr),
                loss='sparse_categorical_crossentropy',
                metrics=['accuracy'])
    return mdl

# ─── Random Search FastText ────────────────────────────────────────────────
FT_GRID = {
    'emb_dim':    [64, 100, 150],
    'ngram_min':  [2, 3],
    'ngram_max':  [3, 4],
    'max_len':    [100, 150, 200],
    'max_feat':   [200000, 500000],
    'lr':         [5e-4, 1e-3, 3e-3],
    'batch_size': [32, 64],
    'epochs':     [30],
}
N_ITER_FT = 10
ft_results = []

for i, p in enumerate(ParameterSampler(FT_GRID, n_iter=N_ITER_FT, random_state=SEED)):
    mn, mx = p['ngram_min'], max(p['ngram_min']+1, p['ngram_max'])
    p['ngram_max'] = mx
    print(f'\n[FastText {i+1}/{N_ITER_FT}] {p}')
    vocab = build_ngram_vocab(X_train_str, mn, mx, p['max_feat'])
    vs    = len(vocab)
    Xtr_f = prepare_ft_seqs(X_train_str, vocab, mn, mx, p['max_len'])
    Xvl_f = prepare_ft_seqs(X_val_str,   vocab, mn, mx, p['max_len'])
    mdl   = build_fasttext_model(vs, p['emb_dim'], N_CLASSES, p['lr'], p['max_len'])
    cb_es = callbacks.EarlyStopping(monitor='val_loss', patience=5,
                                    restore_best_weights=True, verbose=0)
    h = mdl.fit(Xtr_f, y_train, validation_data=(Xvl_f, y_val),
                epochs=p['epochs'], batch_size=p['batch_size'],
                callbacks=[cb_es], verbose=0)
    best_vl = min(h.history['val_loss'])
    print(f'  => Best Val Loss: {best_vl:.4f}')
    ft_results.append({
        'params': p, 'val_loss': best_vl,
        'vocab': vocab, 'vocab_size': vs
    })

best_ft_run = min(ft_results, key=lambda x: x['val_loss'])
best_fp     = best_ft_run['params']
print(f'\nMejores hiperparámetros FastText: {best_fp}')
print(f'Mejor Val Loss: {best_ft_run["val_loss"]:.4f}')


[FastText 1/10] {'ngram_min': 2, 'ngram_max': 3, 'max_len': 150, 'max_feat': 500000, 'lr': 0.003, 'epochs': 30, 'emb_dim': 150, 'batch_size': 64}
  => Best Val Loss: 1.7068

[FastText 2/10] {'ngram_min': 3, 'ngram_max': 4, 'max_len': 100, 'max_feat': 200000, 'lr': 0.0005, 'epochs': 30, 'emb_dim': 100, 'batch_size': 32}
  => Best Val Loss: 1.4290

[FastText 3/10] {'ngram_min': 2, 'ngram_max': 3, 'max_len': 100, 'max_feat': 500000, 'lr': 0.001, 'epochs': 30, 'emb_dim': 150, 'batch_size': 32}
  => Best Val Loss: 1.5903

[FastText 4/10] {'ngram_min': 2, 'ngram_max': 4, 'max_len': 150, 'max_feat': 200000, 'lr': 0.001, 'epochs': 30, 'emb_dim': 64, 'batch_size': 32}
  => Best Val Loss: 1.7962

[FastText 5/10] {'ngram_min': 2, 'ngram_max': 3, 'max_len': 200, 'max_feat': 200000, 'lr': 0.001, 'epochs': 30, 'emb_dim': 150, 'batch_size': 64}
  => Best Val Loss: 1.8594

[FastText 6/10] {'ngram_min': 3, 'ngram_max': 4, 'max_len': 200, 'max_feat': 200000, 'lr': 0.003, 'epochs': 30, 'emb_dim': 64, 'b

In [31]:
# ─── Entrenamiento final FastText ────────────────────────────────────────────
ftr = best_ft_run
mn, mx = best_fp['ngram_min'], best_fp['ngram_max']
Xtr_f = prepare_ft_seqs(X_train_str, ftr['vocab'], mn, mx, best_fp['max_len'])
Xvl_f = prepare_ft_seqs(X_val_str,   ftr['vocab'], mn, mx, best_fp['max_len'])
Xte_f = prepare_ft_seqs(X_test_str,  ftr['vocab'], mn, mx, best_fp['max_len'])

ft_model = build_fasttext_model(ftr['vocab_size'], best_fp['emb_dim'],
                                N_CLASSES, best_fp['lr'], best_fp['max_len'])
cb_ft = [
    callbacks.EarlyStopping(monitor='val_loss', patience=5,
                            restore_best_weights=True, verbose=1),
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3,
                                min_lr=1e-6, verbose=1),
]
hist_ft = ft_model.fit(
    Xtr_f, y_train, validation_data=(Xvl_f, y_val),
    epochs=best_fp['epochs'], batch_size=best_fp['batch_size'],
    callbacks=cb_ft
)

ft_proba = ft_model.predict(Xte_f, batch_size=best_fp['batch_size'])
ft_preds = np.argmax(ft_proba, axis=1)

compute_metrics(y_test, ft_preds, ft_proba, 'FastText')
plot_training_curves(hist_ft.history, 'FastText')
plot_confusion_matrix(y_test, ft_preds, 'FastText')
plot_roc_pr(y_test, ft_proba, 'FastText')

ft_model.save('saved_models/fasttext_best.keras')
with open('saved_models/fasttext_vocab.pkl', 'wb') as f:
    pickle.dump(ftr['vocab'], f)
json.dump({k: (v if not isinstance(v, list) else v)
           for k, v in best_fp.items()},
          open('saved_models/fasttext_config.json', 'w'), indent=2)
print('FastText guardado.')

Epoch 1/30
55/55 [==============================] - 2s 37ms/step - loss: 3.1709 - accuracy: 0.1646 - val_loss: 3.1591 - val_accuracy: 0.3458 - lr: 5.0000e-04
Epoch 2/30
55/55 [==============================] - 2s 35ms/step - loss: 3.1371 - accuracy: 0.3303 - val_loss: 3.1120 - val_accuracy: 0.3324 - lr: 5.0000e-04
Epoch 3/30
55/55 [==============================] - 2s 35ms/step - loss: 3.0583 - accuracy: 0.3608 - val_loss: 3.0050 - val_accuracy: 0.3941 - lr: 5.0000e-04
Epoch 4/30
55/55 [==============================] - 2s 35ms/step - loss: 2.8994 - accuracy: 0.4028 - val_loss: 2.8226 - val_accuracy: 0.3968 - lr: 5.0000e-04
Epoch 5/30
55/55 [==============================] - 2s 35ms/step - loss: 2.6673 - accuracy: 0.4367 - val_loss: 2.5972 - val_accuracy: 0.4236 - lr: 5.0000e-04
Epoch 6/30
55/55 [==============================] - 2s 34ms/step - loss: 2.4080 - accuracy: 0.4914 - val_loss: 2.3818 - val_accuracy: 0.4772 - lr: 5.0000e-04
Epoch 7/30
55/55 [==============================] - 

FastText guardado.


---
## 6. Comparación final de modelos

In [32]:
results_df = pd.DataFrame(RESULTS).T.reset_index()
results_df.columns = ['Modelo', 'Accuracy', 'Precision', 'Recall', 'F1', 'ROC-AUC']
results_df = results_df.sort_values('F1', ascending=False).reset_index(drop=True)
print(results_df.to_string(index=False))

         Modelo  Accuracy  Precision   Recall       F1  ROC-AUC
        RoBERTa  0.790885   0.803970 0.790885 0.780117 0.966505
Word2Vec+BiLSTM  0.756032   0.745113 0.756032 0.741218 0.954323
         CNN-1D  0.745308   0.724717 0.745308 0.725915 0.963954
       FastText  0.648794   0.639022 0.648794 0.632571 0.922087
 TF-IDF+XGBoost  0.528150   0.462627 0.528150 0.477849 0.813491


In [33]:
# ─── Gráfico comparativo de métricas ────────────────────────────────────────
metrics_long = results_df.melt(
    id_vars='Modelo',
    value_vars=['Accuracy', 'Precision', 'Recall', 'F1', 'ROC-AUC'],
    var_name='Métrica', value_name='Valor'
)
fig_comp = px.bar(
    metrics_long,
    x='Métrica', y='Valor', color='Modelo',
    barmode='group', text_auto='.3f',
    title='Comparación de métricas — Todos los modelos',
    template=TEMPLATE, height=500
)
fig_comp.update_layout(yaxis_range=[0, 1.05])
fig_comp.show()

# ─── Radar chart ────────────────────────────────────────────────────────────
cats = ['Accuracy', 'Precision', 'Recall', 'F1', 'ROC-AUC']
fig_radar = go.Figure()
for _, row in results_df.iterrows():
    vals = [row[c] for c in cats] + [row[cats[0]]]
    fig_radar.add_trace(go.Scatterpolar(
        r=vals, theta=cats + [cats[0]],
        fill='toself', name=row['Modelo']
    ))
fig_radar.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0,1])),
    title='Radar de métricas por modelo',
    template=TEMPLATE, height=500
)
fig_radar.show()

In [40]:
# ─── Definir mejor modelo ────────────────────────────────────────────────────
best_row       = results_df.loc[results_df['F1'].idxmax()]
BEST_MODEL_NAME = best_row['Modelo']
BEST_F1         = best_row['F1']

# ─── Tabla estilizada con ranking ───────────────────────────────────────────
results_styled = results_df.copy()
results_styled.index = range(1, len(results_styled) + 1)
medals = ['1', '2', '3'] + [''] * (len(results_styled) - 3)
results_styled.insert(0, 'Rank', medals)

styled = (
    results_styled
    .style
    .format({
        'Accuracy':  '{:.4f}',
        'Precision': '{:.4f}',
        'Recall':    '{:.4f}',
        'F1':        '{:.4f}',
        'ROC-AUC':   '{:.4f}'
    })
    .background_gradient(subset=['Accuracy', 'Precision', 'Recall', 'F1', 'ROC-AUC'],
                         cmap='RdYlGn',   # rojo → amarillo → verde
                         vmin=0.4, vmax=1.0)
    .set_caption(f'Comparación de modelos NLP — Test Set  |  Mejor modelo: {BEST_MODEL_NAME} (F1={BEST_F1:.4f})')
    .set_table_styles([{
        'selector': 'caption',
        'props': [('font-size', '14px'), ('font-weight', 'bold'), ('padding', '10px')]
    }])
)
display(styled)


,Rank,Modelo,Accuracy,Precision,Recall,F1,ROC-AUC
1,1,RoBERTa,0.7909,0.8040,0.7909,0.7801,0.9665
2,2,Word2Vec+BiLSTM,0.7560,0.7451,0.7560,0.7412,0.9543
3,3,CNN-1D,0.7453,0.7247,0.7453,0.7259,0.9640
4,,FastText,0.6488,0.6390,0.6488,0.6326,0.9221
5,,TF-IDF+XGBoost,0.5282,0.4626,0.5282,0.4778,0.8135


---
## 7. Guardado del mejor modelo para producción

Se selecciona el modelo con mayor **F1-score weighted** en test.  
Se genera un **bundle completo** con todo lo necesario para inferencia: modelo, preprocesador, LabelEncoder y metadatos.

In [38]:
BEST_MODEL_NAME = results_df.iloc[0]['Modelo']
BEST_F1         = results_df.iloc[0]['F1']
print(f'Mejor modelo en test: {BEST_MODEL_NAME}  (F1={BEST_F1:.4f})')

# ─── Guardar LabelEncoder (siempre necesario) ────────────────────────────────
joblib.dump(le, 'saved_models/label_encoder.joblib')

# ─── Construir el bundle de producción según el mejor modelo ─────────────────
prod_bundle = {
    'best_model_name':  BEST_MODEL_NAME,
    'best_f1_test':     float(BEST_F1),
    'classes':          list(le.classes_),
    'metrics_all':      RESULTS,
    'preprocessing': {
        'extra_stopwords': list(EXTRA_SW),
        'note': 'Aplicar clean_text() antes de inferencia'
    }
}

if BEST_MODEL_NAME == 'TF-IDF+XGBoost':
    # joblib ya guardado arriba; solo copiamos paths
    prod_bundle['model_path']  = 'saved_models/xgboost_best.joblib'
    prod_bundle['tfidf_path']  = 'saved_models/tfidf_best.joblib'
    prod_bundle['framework']   = 'sklearn+xgboost'
    prod_bundle['load_snippet'] = (
        "import joblib\n"
        "tfidf = joblib.load('saved_models/tfidf_best.joblib')\n"
        "model = joblib.load('saved_models/xgboost_best.joblib')\n"
        "le    = joblib.load('saved_models/label_encoder.joblib')\n"
        "X = tfidf.transform([clean_text(raw_text)])\n"
        "pred  = le.inverse_transform(model.predict(X))"
    )
elif BEST_MODEL_NAME == 'Word2Vec+BiLSTM':
    prod_bundle['model_path']  = 'saved_models/bilstm_best.keras'
    prod_bundle['vocab_path']  = 'saved_models/bilstm_word2idx.pkl'
    prod_bundle['seq_len']     = best_bp['seq_len']
    prod_bundle['framework']   = 'tensorflow+keras'
    prod_bundle['load_snippet'] = (
        "import pickle, numpy as np\n"
        "from tensorflow.keras.models import load_model\n"
        "model   = load_model('saved_models/bilstm_best.keras')\n"
        "word2idx= pickle.load(open('saved_models/bilstm_word2idx.pkl','rb'))\n"
        "le      = joblib.load('saved_models/label_encoder.joblib')\n"
        "tokens  = clean_text(raw_text).split()\n"
        "seq     = build_sequences([tokens], word2idx, seq_len)\n"
        "pred    = le.inverse_transform([np.argmax(model.predict(seq))])"
    )
elif BEST_MODEL_NAME == 'CNN-1D':
    prod_bundle['model_path']  = 'saved_models/cnn1d_best.keras'
    prod_bundle['vocab_path']  = 'saved_models/cnn1d_vocab.pkl'
    prod_bundle['config_path'] = 'saved_models/cnn1d_config.json'
    prod_bundle['framework']   = 'tensorflow+keras'
elif BEST_MODEL_NAME == 'FastText':
    prod_bundle['model_path']  = 'saved_models/fasttext_best.keras'
    prod_bundle['vocab_path']  = 'saved_models/fasttext_vocab.pkl'
    prod_bundle['config_path'] = 'saved_models/fasttext_config.json'
    prod_bundle['framework']   = 'tensorflow+keras'
elif BEST_MODEL_NAME == 'RoBERTa':
    prod_bundle['model_path']  = 'saved_models/roberta_best.weights.h5'
    prod_bundle['base_model']  = 'roberta-base'
    prod_bundle['max_len']     = best_roberta_cfg['max_len']
    prod_bundle['framework']   = 'transformers+tensorflow'

# ─── Guardar metadatos del bundle ────────────────────────────────────────────
with open('saved_models/production_bundle.json', 'w') as f:
    json.dump(prod_bundle, f, indent=2)

print('\n=== Bundle de producción guardado ===')
print(json.dumps({k: v for k, v in prod_bundle.items()
                  if k != 'metrics_all'}, indent=2))

Mejor modelo en test: RoBERTa  (F1=0.7801)

=== Bundle de producción guardado ===
{
  "best_model_name": "RoBERTa",
  "best_f1_test": 0.7801170031340156,
  "classes": [
    "ACCOUNTANT",
    "ADVOCATE",
    "AGRICULTURE",
    "APPAREL",
    "ARTS",
    "AUTOMOBILE",
    "AVIATION",
    "BANKING",
    "BPO",
    "BUSINESS-DEVELOPMENT",
    "CHEF",
    "CONSTRUCTION",
    "CONSULTANT",
    "DESIGNER",
    "DIGITAL-MEDIA",
    "ENGINEERING",
    "FINANCE",
    "FITNESS",
    "HEALTHCARE",
    "HR",
    "INFORMATION-TECHNOLOGY",
    "PUBLIC-RELATIONS",
    "SALES",
    "TEACHER"
  ],
  "preprocessing": {
    "extra_stopwords": [
      "name",
      "company",
      "city",
      "skills",
      "state",
      "experience"
    ],
    "note": "Aplicar clean_text() antes de inferencia"
  },
  "model_path": "saved_models/roberta_best.weights.h5",
  "base_model": "roberta-base",
  "max_len": 256,
  "framework": "transformers+tensorflow"
}


---
## 8. Pipeline de inferencia — Cómo usar el mejor modelo

Este ejemplo carga el mejor modelo y clasifica un CV nuevo en una sola línea.

In [39]:
# ─── Cargar bundle de producción ────────────────────────────────────────────
bundle = json.load(open('saved_models/production_bundle.json'))
le_inf = joblib.load('saved_models/label_encoder.joblib')

def predict_resume(raw_text: str) -> str:
    """
    Clasifica un CV en texto plano y devuelve la categoría laboral.
    Usa automáticamente el mejor modelo entrenado.
    """
    model_name = bundle['best_model_name']
    text_clean = clean_text(raw_text)

    if model_name == 'TF-IDF+XGBoost':
        tfidf_inf = joblib.load(bundle['tfidf_path'])
        xgb_inf   = joblib.load(bundle['model_path'])
        X   = tfidf_inf.transform([text_clean])
        pred = xgb_inf.predict(X)[0]

    elif model_name in ('Word2Vec+BiLSTM', 'CNN-1D'):
        from tensorflow.keras.models import load_model as lm
        with open(bundle['vocab_path'], 'rb') as f:
            vocab_inf = pickle.load(f)
        if model_name == 'CNN-1D':
            cfg_inf = json.load(open(bundle['config_path']))
            seq_len_inf = cfg_inf['max_len']
        else:
            seq_len_inf = bundle['seq_len']
        keras_mdl = lm(bundle['model_path'])
        tokens    = text_clean.split()
        seq       = build_sequences([tokens], vocab_inf, seq_len_inf)
        proba     = keras_mdl.predict(seq, verbose=0)
        pred      = int(np.argmax(proba, axis=1)[0])

    elif model_name == 'FastText':
        from tensorflow.keras.models import load_model as lm
        with open(bundle['vocab_path'], 'rb') as f:
            vocab_inf = pickle.load(f)
        cfg_inf   = json.load(open(bundle['config_path']))
        keras_mdl = lm(bundle['model_path'])
        seq = prepare_ft_seqs([text_clean], vocab_inf,
                               cfg_inf['ngram_min'], cfg_inf['ngram_max'],
                               cfg_inf['max_len'])
        proba = keras_mdl.predict(seq, verbose=0)
        pred  = int(np.argmax(proba, axis=1)[0])

    elif model_name == 'RoBERTa':
        rob_mdl = TFRobertaForSequenceClassification.from_pretrained(
            bundle['base_model'], num_labels=N_CLASSES
        )
        rob_mdl.load_weights(bundle['model_path'])
        enc  = roberta_tokenizer([raw_text], max_length=bundle['max_len'],
                                  padding='max_length', truncation=True,
                                  return_tensors='tf')
        logits = rob_mdl(enc, training=False).logits
        pred   = int(np.argmax(tf.nn.softmax(logits).numpy(), axis=1)[0])

    return le_inf.inverse_transform([pred])[0]


# ─── Prueba rápida ────────────────────────────────────────────────────────
cv_ejemplo = """
Experienced Data Scientist with 5 years in machine learning, deep learning,
Python, TensorFlow. Led projects on NLP and recommendation systems.
MSc Computer Science, Stanford University.
"""
resultado = predict_resume(cv_ejemplo)
print(f'Categoría predicha: {resultado}')

Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFRobertaForSequenceClassification: ['roberta.embeddings.position_ids']
- This IS expected if you are initializing TFRobertaForSequenceClassification from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFRobertaForSequenceClassification from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
Some weights or buffers of the TF 2.0 model TFRobertaForSequenceClassification were not initialized from the PyTorch model and are newly initialized: ['classifier.dense.weight', 'classifier.dense.bias', 'classifier.out_proj.weight', 'classifier.out_proj.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predicti

Categoría predicha: INFORMATION-TECHNOLOGY


---
## 9. Resumen de archivos guardados

| Archivo | Contenido | Cómo cargar |
|---|---|---|
| `saved_models/label_encoder.joblib` | Codificador de etiquetas | `joblib.load(...)` |
| `saved_models/tfidf_best.joblib` | Vectorizador TF-IDF | `joblib.load(...)` |
| `saved_models/xgboost_best.joblib` | Modelo XGBoost | `joblib.load(...)` |
| `saved_models/bilstm_best.keras` | Red BiLSTM | `load_model(...)` |
| `saved_models/bilstm_word2idx.pkl` | Vocabulario BiLSTM | `pickle.load(...)` |
| `saved_models/cnn1d_best.keras` | Red CNN-1D | `load_model(...)` |
| `saved_models/cnn1d_vocab.pkl` | Vocabulario CNN-1D | `pickle.load(...)` |
| `saved_models/cnn1d_config.json` | Hiperparámetros CNN-1D | `json.load(...)` |
| `saved_models/fasttext_best.keras` | Red FastText | `load_model(...)` |
| `saved_models/fasttext_vocab.pkl` | Vocabulario n-gramas | `pickle.load(...)` |
| `saved_models/fasttext_config.json` | Config FastText | `json.load(...)` |
| `saved_models/roberta_best.weights.h5` | Pesos RoBERTa | `model.load_weights(...)` |
| `saved_models/production_bundle.json` | Metadatos del mejor modelo | `json.load(...)` |

### ¿Por qué distintos formatos?
- **joblib** → modelos sklearn-compatible (TF-IDF, XGBoost): serialización eficiente para objetos Python grandes con arrays numpy.
- **`.keras`** → modelos Keras/TensorFlow: formato nativo que guarda arquitectura + pesos + optimizador.
- **`.weights.h5`** → solo pesos (RoBERTa): se guarda solo pesos porque la arquitectura se reconstruye desde `from_pretrained`.
- **pickle** → vocabularios (diccionarios Python puros).
- **json** → configuraciones e hiperparámetros (legibles por humanos y cualquier lenguaje).